# piml-feature-engines: run every experiment

Runs every experiment of the repository from scratch on Google Colab. Run the cells in order (*Runtime › Run all*). Every script checkpoints and resumes; with `SAVE_TO_DRIVE = True` results are copied to Google Drive after each step. Set `SMOKE = True` for a ten-minute end-to-end check.

| Step | What it computes | Time (Colab CPU) |
|---|---|---|
| setup | configuration, code and data | 5 min |
| theory checks | collapse of the LG descriptors, identifiability, Kramers regime, energy landscape | 3 min |
| C-MAPSS | grouped five-fold CV of all feature arms, rate arms and controls, supervised multi-mode | 4–5 h |
| Azure and batteries | the same comparison with a budget-matched baseline | 30 min |
| order-parameter channels | the three unsupervised channels on FD003 | 1 min |
| critical slowing down | test with a memoryless null model | 2 min |
| turning simulation | known Hopf boundary, model-based estimators | 1–1.5 h |
| robustness | IBS, time-dependent AUC, remaining-life error | 30 min |
| case study | warning zones and lead times | 1 min |
| cost | per-sample inference cost | 5 min |
| Cox with a Kramers term | against a free spline | 10 min |

In [ ]:
# 1. Configuration
REPO_URL      = 'https://github.com/diogrocha/piml-feature-engines.git'
BRANCH        = 'main'
SMOKE         = False
SAVE_TO_DRIVE = True
DRIVE_DIR     = '/content/drive/MyDrive/lg_features_results'
SUBSETS       = ['FD001', 'FD003', 'FD002', 'FD004']
FOLDS         = '1' if SMOKE else '1,2,3,4,5'

In [ ]:
# 2a. Code
import os, subprocess, glob
def run(cmd, check=True):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='')
    p.wait()
    if check and p.returncode != 0: raise RuntimeError(f'failed ({p.returncode}): {cmd}')
os.chdir('/content')
if not os.path.isdir('/content/repo'): run(f'git clone --depth 1 -b {BRANCH} {REPO_URL} repo')
os.chdir('/content/repo'); run('pip -q install -r requirements.txt'); os.makedirs('results', exist_ok=True)
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive'); os.makedirs(DRIVE_DIR, exist_ok=True)
def save():
    if SAVE_TO_DRIVE: run(f'cp -r results/. "{DRIVE_DIR}/"', check=False)

In [ ]:
# 2b. Data (public mirrors of the official files, checked by line count)
run('bash get_data.sh')
CM = 'data/CMAPSSData'
for f in [f'test_FD00{i}.txt' for i in range(1, 5)] + [f'RUL_FD00{i}.txt' for i in range(1, 5)]:
    if not os.path.exists(f'{CM}/{f}'):
        run(f'curl -sfL -o {CM}/{f} https://raw.githubusercontent.com/schwxd/LSTM-Keras-CMAPSS/master/C-MAPSS-Data/{f}')
os.makedirs('data/azure', exist_ok=True)
for f in ['PdM_telemetry.csv', 'PdM_maint.csv', 'PdM_failures.csv', 'PdM_errors.csv', 'PdM_machines.csv']:
    if not os.path.exists(f'data/azure/{f}'):
        run(f'curl -sfL -o data/azure/{f} https://raw.githubusercontent.com/l1h2/predictive_maintenance_on_azure_dataset/main/data/{f}')
os.makedirs('data/nasa_battery', exist_ok=True)
for b in ['B0005', 'B0006', 'B0007', 'B0018']:
    if not os.path.exists(f'data/nasa_battery/{b}.mat'):
        run(f'curl -sfL -o data/nasa_battery/{b}.mat https://raw.githubusercontent.com/Hankxu-316/Visual-analysis-system-for-lithium-battery-health-status/main/data/{b}.mat')
for f, n in {f'{CM}/train_FD001.txt': 20631, f'{CM}/train_FD002.txt': 53759, f'{CM}/train_FD003.txt': 24720, f'{CM}/train_FD004.txt': 61249, f'{CM}/test_FD001.txt': 13096}.items():
    k = sum(1 for _ in open(f)); print(f'{f:<30} {k:6d} lines', 'OK' if k == n else f'EXPECTED {n}')
os.environ['AZURE_DATA'] = '/content/repo/data/azure'

## Theory checks

In [ ]:
run('python theory_checks/apriori_checks.py' + (' --nfast 600' if SMOKE else '')); run('python theory_checks/landscape_figure.py'); save()

## C-MAPSS, all feature arms

In [ ]:
run('python cmapss_paradigms/run_cv.py' + (' --smoke --out results/smoke.json' if SMOKE else ''))
if not SMOKE: run('python cmapss_paradigms/summarize.py')
for s in SUBSETS:
    run(f'python cmapss_paradigms/lg_comb_gbsa.py {s} {FOLDS} LG-comb-mm,PC-twin,HI-fix'); save()
run(f'python cmapss_paradigms/sup_mm.py FD003 {FOLDS}'); save()

## Azure and NASA batteries

In [ ]:
run('python src/azure_run.py prep comp1 comp2 comp3 comp4')
run('python azure_batteries/other_datasets_matched.py azure --arms Base Base-AUG +Stats +HI +Weibull +Entropy +ExpDeg +LG')
run('python azure_batteries/extra_arms.py azure --cache /tmp/azure_cache')
run('python azure_batteries/other_datasets_matched.py battery --data /content/repo/data/nasa_battery')
run('python azure_batteries/extra_arms.py battery --data /content/repo/data/nasa_battery')
save()

## Order-parameter channels

In [ ]:
run('python mode_channels/plot_mode_channels.py'); save()

## Critical-slowing-down test

In [ ]:
run('python critical_slowing_down/csd_audit.py FD001 FD003'); save()

## Turning simulation
Regenerative turning with a slow ramp of the depth of cut; no external data. The fleet passes run in parallel over seed ranges.

In [ ]:
os.chdir('/content/repo/turning_simulation')
seeds = [(0, 20), (20, 40), (40, 60), (60, 80)] if not SMOKE else [(0, 4)]
run('python -c "import fleet_windows"')          # builds the stability-limit table once
for p in ['fleet_windows.py', 'fleet_recursive.py', 'fleet_landau.py']:
    run(' & '.join(f'python {p} {a} {b}' for a, b in seeds) + ' & wait')
run('python merge.py')
run('python paradigms.py Base,+Stats,+HI,+HI-fix,+Weibull,+Entropy,+ExpDeg,+LG,+LG-4,+LG-mm,LG-comb-mm,PC-twin,+AR-raw,+b_ls,+d_hat,+twin-est,oracle')
run('python estimators_recursive.py Base,+b_ls,+b_rls,+d_rls,+full-twin,+sigma_L,+Landau-twin,+b_rls+sigma_L,oracle')
run('python estimators_landau.py Base,+d_rls,+sigma_L2,+T_full,+twin_full,+T_L,+twin_L,oracle')
run('python plot_simulation.py')
run('mkdir -p /content/repo/results/sim && cp paradigms_sim.json survival3.json survival4.json fig_turning_sim.pdf /content/repo/results/sim/')
os.chdir('/content/repo'); save()

## Robustness

In [ ]:
if not SMOKE: run('python robustness/robustness.py')
save()

## Case study

In [ ]:
run('python case_study/case_study.py'); save()

## Inference cost

In [ ]:
run('python cost/cost_e2e.py'); save()

## Cox model with a Kramers-type term

In [ ]:
for s in SUBSETS:
    run(f'python cox_kramers/kramers_cox.py {s} {FOLDS}')
save()

## Results archive

In [ ]:
run('zip -qr /content/lg_features_results.zip results')
from google.colab import files; files.download('/content/lg_features_results.zip')